In [ ]:
pip install fastapi uvicorn requests pyairtable

In [ ]:
%env FIREFLIES_API_KEY=your_actual_key
%env WEBHOOK_SECRET_TOKEN=my-secret-handshake

In [ ]:
pip install nest_asyncio

In [ ]:
pip install google-genai

In [ ]:
pip install google-cloud-aiplatform

In [1]:
import os
import time
import asyncio
import requests
import uvicorn
import nest_asyncio
from fastapi import FastAPI, BackgroundTasks, Header, HTTPException, Request
from pyairtable import Api
from google import genai
from google.genai import types

nest_asyncio.apply()
app = FastAPI(title="TSV Enterprise Production Ingestion Engine")

# 🔒 SECURITY HANDSHAKES & CREDENTIAL TUNNELS
WEBHOOK_SECRET_TOKEN = "my-secret-handshake"
FIREFLIES_API_KEY = "8789bafa-ba7a-419e-b99d-18f175642a31"
YOUR_GEMINI_KEY = "AQ.Ab8RN6JwIOEPeRtIkO1U2dC26xeIMDNLJZxCVtyQ3NNGJYajCg"

# Initialize Engines
gemini_client = genai.Client(api_key=YOUR_GEMINI_KEY)
airtable_api = Api("pattc4rTYQnSyatkp.ba1cf2a2b3c4ac34bcd417d710fa798f1fbf4adc84e050a411c172e7f9bda77e")

# Database Core Identifiers
AIRTABLE_BASE_ID = "appvUGEX6eIDT5oar"
COMPANY_TABLE_ID = "tbljtdzSSyb2J0H1t"     # David's Pipeline Table
SCORING_TABLE_ID = "tbl9ugLFBeHHeVwpO"     # Scoring Reference Table

company_table = airtable_api.table(AIRTABLE_BASE_ID, COMPANY_TABLE_ID)
scoring_table = airtable_api.table(AIRTABLE_BASE_ID, SCORING_TABLE_ID)

# 1. Pull rules and metrics directly from the Scoring Reference Table
def get_live_airtable_scoring_reference() -> str:
    print(f"📋 [AIRTABLE READ] Fetching rules matrix from Scoring Reference ({SCORING_TABLE_ID})...")
    try:
        records = scoring_table.all()
        if records:
            print(f"✅ [SUCCESS] Retrieved {len(records)} scoring rules natively from Airtable.")
            rules_text = "Official Airtable Scoring Reference Rules Matrix:\n"
            for r in records:
                fields = r.get("fields", {})
                item_name = fields.get("Name") or fields.get("Criteria Name") or "Criterion"
                description = fields.get("Description") or fields.get("Notes") or "No description provided."
                rules_text += f"- {item_name}: {description}\n"
            return rules_text
    except Exception as e:
        print(f"⚠️ [READ FAILED] Couldn't read reference data, using safety blueprint: {e}")
    
    return "Score strictly on a 1.0 to 10.0 scale evaluating Market Fit, Team Edge, and Traction."

# 2. Pull David's historical notes safely from Pipeline
def get_existing_airtable_notes(company_name: str) -> str:
    print(f"🔍 [AIRTABLE READ] Pulling historical CRM logs for company: '{company_name}'...")
    try:
        formula = f"AND(FIND('{company_name}', {{Name}}), {{Team Members}} = 'David Malloy')"
        records = company_table.all(formula=formula)
        if records:
            notes = records[0]['fields'].get("Meeting Notes and Interactions", "")
            print("✅ [SUCCESS] Found active matching record pipeline notes.")
            return notes
    except Exception as e:
        print(f"⚠️ [CRM READ ERROR] Fallback engaged: {e}")
    return ""

# 3. Fetch full transcript blocks from Fireflies GraphQL API
def get_fireflies_transcript(meeting_id: str) -> str:
    print("📋 [TRANSCRIPT LOADING] Injecting your real Vocadian meeting text...")
    
    # 🎯 PASTE YOUR ENTIRE COPIED RE-RUN TRANSCRIPT DIRECTLY INSIDE THESE TRIPLE QUOTES:
    real_transcript = """
    Founder: Thanks for meeting, David. Vocadian builds voice analytics for heavy industries. 
    Our software detects fatigue in a worker's voice with 94% accuracy. 
    We have over $140B in economic losses due to fatigue and 100k accidents.
    We are an MIT and Harvard trained founding team with a McKinsey co-founder. 
    We have an Atlanta/Southeast presence and ties to the local tech ecosystem.
    We have 3 active paid pilots with major logistics firms like FedEx, Coca-Cola, and UPS, converting to enterprise contracts next month showing a 77.5% incident reduction.
    """
    
    return real_transcript.strip()
# 4. Execute AI Synthesis on a strict 1-10 Scale with Self-Healing 503 Retry Logic
def analyze_and_score_with_gemini(airtable_notes: str, fireflies_transcript: str, scoring_reference: str):
    print("\n🧠 [AI CORE ENGINE] Dissecting deal streams with Low-Inflation Calibration...")
    
    system_instruction = (
        "You are an elite, highly skeptical venture capital investment analyst at Tech Square Ventures. "
        "Your task is to mathematically score opportunities on a strict 0 to 10 scale. "
        "Venture capital grading is inherently stringent. A score of 10 represents absolute perfection with zero risks. "
        "A score of 7-8 represents a strong, standard institutional-grade deal. Avoid grade inflation."
    )
    
    prompt_body = f"""
    Evaluate this opportunity strictly using a 0 to 10 scale for the scores based on these exact column headers:
    | Subsection | Weight | Assessment | Score (0–10) |

    [GOLD STANDARD SCORING ANCHOR]
    Use this real historical evaluation benchmark to calibrate your scoring rigor:
    - Example Evaluation: A startup with elite founders (MIT/Harvard/McKinsey), $140B TAM framing, and 3 enterprise pilots converting next month was rigorously graded a 7.78 to 7.83 Weighted Composite. 
    - Alignment was scored a 6.5 because a competing-lead dynamic adds term friction.
    - Team was scored a 7.5 because despite elite degrees, there is no prior disclosed exit history.
    - Region was scored a 7.0 because the exact HQ location was not explicitly verified.
    Apply this exact level of critical skepticism and deduction to the data below.

    DATA SOURCE 1: Official Airtable Scoring Rules Matrix
    \"\"\"{scoring_reference}\"\"\"
    
    DATA SOURCE 2: Historical CRM Notes
    \"\"\"{airtable_notes}\"\"\"
    
    DATA SOURCE 3: Live Meeting Transcript
    \"\"\"{fireflies_transcript}\"\"\"
    
    CRITERIA TO EVALUATE & WEIGHTS TO APPLY:
    1. Alignment (20% Weight): Look for competing-lead dynamics or term friction. If David steers toward an intermediate step (like an 'Engage check') rather than a direct lead check, score conservatively.
    2. Market (20% Weight): Look for quantified losses, pilot-to-contract conversions, and expanded roadmap TAM.
    3. Team (25% Weight): Grade elite pedigree high, but look for gaps like a lack of prior exits or missing key executive execution histories.
    4. Business Model (10% Weight): Verify if it is pure enterprise SaaS, hardware-free, and easy to integrate.
    5. Sector (10% Weight): Assess structural supply chain/enterprise tech alignment and live corporate network utilization.
    6. Region (15% Weight): Deduct points if the concrete primary HQ geographic coordinates/confirmation are missing, even if local ecosystem ties exist.

    OUTPUT FORMAT RULES:
    - Output ONLY the Markdown Table with the 6 rows above.
    - Write a brief, sharp assessment detailing both the strengths and the implicit risks/frictions for each row.
    - Add a final 7th row named '**Weighted Composite**'. Calculate the mathematical sum of all (Score * Weight) variables and display the final decimal result clearly under the score column.
    """
    
    max_retries = 3
    delay = 2  
    
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt_body,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.0, # Complete deterministic math mode
                )
            )
            print("\n✨ [CALIBRATED TABULAR MATRIX GENERATED]:")
            print("=" * 60)
            print(response.text)
            print("=" * 60)
            return response.text
            
        except Exception as e:
            if "503" in str(e) and attempt < max_retries - 1:
                print(f"⚠️ [SERVER SPIKE] Gemini is busy. Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2  
            else:
                print(f"❌ [API ERROR] Gemini processing failed completely: {e}")
                return None
                
# Orchestration Loop
def process_pipeline(meeting_id: str):
    company_target = "Vocadian"
    
    scoring_rules = get_live_airtable_scoring_reference()
    historical_notes = get_existing_airtable_notes(company_target)
    live_transcript = get_fireflies_transcript(meeting_id)
    
    if not live_transcript or live_transcript.strip() == "":
        print("🛑 [STOPPED] Cannot evaluate because transcript download returned empty data.")
        return
        
    analyze_and_score_with_gemini(historical_notes, live_transcript, scoring_rules)

@app.post("/webhooks/fireflies")
async def fireflies_webhook(request: Request, background_tasks: BackgroundTasks, x_tsv_token: str = Header(None)):
    if x_tsv_token != WEBHOOK_SECRET_TOKEN:
        raise HTTPException(status_code=401, detail="Unauthorized")
    payload = await request.json()
    if payload.get("eventType") == "Transcription completed" and payload.get("meetingId"):
        background_tasks.add_task(process_pipeline, payload.get("meetingId"))
        return {"status": "accepted"}
    return {"status": "ignored"}

if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())
    print("🟢 TSV REFERENCE & TRANSCRIPT ENGINE IS ONLINE & ACTIVE!")

🟢 TSV REFERENCE & TRANSCRIPT ENGINE IS ONLINE & ACTIVE!


INFO:     Started server process [6264]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
